# Exploración: Identidad y Padrón Fiscal (apis.net.pe)

- **Portal / Proveedor:** `https://apis.net.pe`
- **Mecanismo:** Servicio REST JSON vía peticiones HTTP GET directas.

- **Restricción de parámetros:**
  - Consulta RUC: Exclusivamente parámetro numérico exacto de 11 dígitos (`numero`). No admite razón social.
  - Consulta DNI: Exclusivamente parámetro numérico exacto de 8 dígitos (`numero`). No admite nombres ni apellidos.
- **Endpoints evaluados:**
  - `GET /v1/ruc?numero={ruc}`: Devuelve razón social, estado (ACTIVO, SUSPENSIÓN TEMPORAL, BAJA), condición (HABIDO / NO HABIDO) y domicilio fiscal normalizado (departamento, provincia, distrito, ubigeo).
  - `GET /v1/dni?numero={dni}`: Devuelve nombre completo, desglose de apellidos y nombres para personas naturales.
- **Destino en arquitectura:** Adaptador complementario para enriquecer la ficha básica del investigado antes o después de la trazabilidad SEACE CDA.

In [5]:
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any

# Parámetros de prueba
DNI_OBJETIVO = "44900952"
RUC_OBJETIVO = "20602102344"

# Parámetros para pruebas de validación
DNI_INVALIDO = "00000000"
RUC_INEXISTENTE = "20999999999"

# Configuración HTTP base
BASE_URL = "https://api.apis.net.pe/v1"
HEADERS_APIS_NET = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Referer": "https://apis.net.pe/"
}

print("Variables y cabeceras configuradas correctamente.")

Variables y cabeceras configuradas correctamente.


In [6]:
async def consultar_dni(dni: str) -> Dict[str, Any]:
    url = f"{BASE_URL}/dni"
    params = {"numero": dni.strip()}
    inicio = datetime.now(timezone.utc)
    
    async with httpx.AsyncClient(timeout=15.0, headers=HEADERS_APIS_NET) as client:
        try:
            resp = await client.get(url, params=params)
            duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)
            
            if resp.status_code == 200:
                raw = resp.json()
                return {
                    "audit": {
                        "source": "APIS_NET_PE_DNI",
                        "tax_id": dni,
                        "status_code": 200,
                        "duration_ms": duracion,
                        "success": True
                    },
                    "data": {
                        "document_type": "DNI",
                        "document_number": raw.get("numeroDocumento"),
                        "full_name": raw.get("nombre"),
                        "first_surname": raw.get("apellidoPaterno"),
                        "second_surname": raw.get("apellidoMaterno"),
                        "names": raw.get("nombres")
                    },
                    "raw": raw
                }
            return {
                "audit": {"source": "APIS_NET_PE_DNI", "tax_id": dni, "status_code": resp.status_code, "duration_ms": duracion, "success": False},
                "data": None,
                "raw": resp.text[:200]
            }
        except Exception as exc:
            return {
                "audit": {"source": "APIS_NET_PE_DNI", "tax_id": dni, "status_code": None, "duration_ms": None, "success": False},
                "data": None,
                "error": str(exc)
            }

# Ejecutar prueba de DNI
res_dni = await consultar_dni(DNI_OBJETIVO)
print(f"Status: {res_dni['audit']['status_code']} | Duración: {res_dni['audit']['duration_ms']} ms")
print(json.dumps(res_dni["data"], indent=2, ensure_ascii=False))

Status: 200 | Duración: 473 ms
{
  "document_type": "DNI",
  "document_number": "44900952",
  "full_name": "RONALD RUIZ AUCCAPIÑA",
  "first_surname": "RUIZ",
  "second_surname": "AUCCAPIÑA",
  "names": "RONALD"
}


In [7]:
async def consultar_ruc(ruc: str) -> Dict[str, Any]:
    url = f"{BASE_URL}/ruc"
    params = {"numero": ruc.strip()}
    inicio = datetime.now(timezone.utc)
    
    async with httpx.AsyncClient(timeout=15.0, headers=HEADERS_APIS_NET) as client:
        try:
            resp = await client.get(url, params=params)
            duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)
            
            if resp.status_code == 200:
                raw = resp.json()
                return {
                    "audit": {
                        "source": "APIS_NET_PE_RUC",
                        "tax_id": ruc,
                        "status_code": 200,
                        "duration_ms": duracion,
                        "success": True
                    },
                    "data": {
                        "document_type": "RUC",
                        "document_number": raw.get("numeroDocumento"),
                        "business_name": raw.get("nombre"),
                        "status": raw.get("estado"),
                        "condition": raw.get("condicion"),
                        "address": raw.get("direccion"),
                        "ubigeo": raw.get("ubigeo"),
                        "department": raw.get("departamento"),
                        "province": raw.get("provincia"),
                        "district": raw.get("distrito")
                    },
                    "raw": raw
                }
            return {
                "audit": {"source": "APIS_NET_PE_RUC", "tax_id": ruc, "status_code": resp.status_code, "duration_ms": duracion, "success": False},
                "data": None,
                "raw": resp.text[:200]
            }
        except Exception as exc:
            return {
                "audit": {"source": "APIS_NET_PE_RUC", "tax_id": ruc, "status_code": None, "duration_ms": None, "success": False},
                "data": None,
                "error": str(exc)
            }

# Ejecutar prueba de RUC
res_ruc = await consultar_ruc(RUC_OBJETIVO)
print(f"Status: {res_ruc['audit']['status_code']} | Duración: {res_ruc['audit']['duration_ms']} ms")
print(json.dumps(res_ruc["data"], indent=2, ensure_ascii=False))

Status: 200 | Duración: 461 ms
{
  "document_type": "RUC",
  "document_number": "20602102344",
  "business_name": "LM GLOBAL IMPORT EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA",
  "status": "SUSPENSION TEMPORAL",
  "condition": "HABIDO",
  "address": "MZA. E7 LOTE 10 URB. LARAPA GRANDE ",
  "ubigeo": "080104",
  "department": "CUSCO",
  "province": "CUSCO",
  "district": "SAN JERONIMO"
}
